In [0]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, auc, precision_recall_curve, confusion_matrix, classification_report, ConfusionMatrixDisplay, make_scorer
from sklearn.base import clone
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from chart_utils import set_pomegranate_theme, POMEGRANATE_PALETTE, plot_100pct_stacked_hbar, plot_100pct_stacked_hbar_by_group, plot_distribution_bar, show_plot
set_pomegranate_theme()

Constants from the original data description will be used to calculate estimated profit.  The original description explains that every cow represents a customer who received the latest offer, and that '\[the\] total cost of the sample campaign was 6.720MU and the revenue generated by the customers who accepted the offer was 3.674MU. Globally the campaign had a profit of -3.046MU'.  Although I've dropped rows in the cleaning process, we'll use the original counts to estimate the cost per offer and revenue per acceptance (note that this results very cleanly to 3.00 and 11.00). Assume that '.' is a thousands-separator as in Brazil and that 'MU' stands for 'monetary units'.


In [0]:
TOTAL_COST = 6720
TOTAL_REVENUE = 3674
TOTAL_ROWS = 2240
TOTAL_ACCEPTED = 334

COST_PER_OFFER = TOTAL_COST / TOTAL_ROWS
REVENUE_PER_CONV = TOTAL_REVENUE / TOTAL_ACCEPTED

print(f"Cost per Offer: {COST_PER_OFFER:.2f} | Revenue per Conv: {REVENUE_PER_CONV:.2f}")


The business goal is to make offers to customers who are likely to accept in such a way as to *maximise net profit*; making an unaccepted offers has a cost.  Therefore, create functions to calculate net profit for custom scoring.  The default cost per offer and revenue per acceptance are here set to this campaign's stated cost and revenue mentioned above.

In [0]:
def calculate_net_profit(y_true, y_pred_proba, threshold=0.5, cost_offer=None, revenue_accept=None):
    if cost_offer is None: cost_offer = COST_PER_OFFER
    if revenue_accept is None: revenue_accept = REVENUE_PER_CONV
    y_pred = (y_pred_proba >= threshold).astype(int)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    all_offers = np.sum(y_pred == 1)
    # Net profit = (TP * Revenue) - (all_offers * Cost).  TN and FN have 0 cost and 0 revenue
    profit = (tp * revenue_accept) - (all_offers * cost_offer)
    return profit

def custom_profit_scorer(estimator, X, y):
    # Wrapper for cross-validation.  Finds the optimal threshold for each fold to maximize profit
    probs = estimator.predict_proba(X)[:, 1]
    
    # Search for the optimal threshold on this specific validation fold, iterating through range of thresholds
    thresholds = np.arange(0.1, 0.91, 0.01)
    best_profit = -np.inf
    best_threshold = 0.5
    
    for thresh in thresholds:
        p = calculate_net_profit(y, probs, threshold=thresh)
        if p > best_profit:
            best_profit = p
            best_threshold = thresh
            
    return best_profit

Load and prepare the dataset.  It's already been cleaned for EDA and segmentation, but we have just over 1% of rows with imputed Age and/or Income, so we will drop those rows, then select the desired features.  I tested two approaches: 
* near-raw features (using numeric equivalents for raw date and category fields) with automatic selection of most important fields from a baseline RandomForests model
* selection of raw and engineered features capturing customer spending and engagement patterns, although still dropping highly-correlated (> 0.85) fields

The latter resulted in a significantly higher-scoring model and is used here.

In [0]:
base_dir = Path.cwd()  
output_dir = base_dir.parent / "output"
csv_path = base_dir.parent / "data" / "processed" / "marketing_clean.csv"
marketing_clean = pd.read_csv(csv_path)
marketing_clean = marketing_clean[(marketing_clean['has_imputed_income'] == 0) & (marketing_clean['has_imputed_age'] == 0)]

# Target variable
y = marketing_clean['Response'].copy()

near_raw_features = ['Income', 'Kidhome', 'Teenhome', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts',
       'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases',
       'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'DaysSinceJoin', 'Complain',  'Education_years', 'Age','Adulthome']

manual_features = ['TotalSpend', 'SpendPerMonth', 'SpendPerPurchase', 'NumWebVisitsMonth', 'NumWebPurchases',
                   'NumCatalogPurchases', 'NumStorePurchases', 'PercentGold', 'PercentDealPurchases', 'Recency', 'DaysSinceJoin','Education_years', 'Age', 'Income', 'Minorhome', 'Adulthome', 'Complain', 'NumAccepted']

# Features
X = marketing_clean[manual_features].copy()
print(X.shape)
na_counts = X.isna().sum()
missing = na_counts[na_counts > 0]
if missing.empty:
    print("No missing values.")
else:
    print("Missing values found:")
    print(missing)
print(f"Target distribution:\n{y.value_counts(normalize=True).map('{:.2%}'.format)}")

Train-test split.

In [0]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train Shape: {X_train_full.shape}, Test Shape: {X_test.shape}")
print(f"Train Response Rate: {y_train_full.mean():.2%}, Test Response Rate: {y_test.mean():.2%}")


Remove highly correlated features (using train data only), then apply same drop to test data.

In [0]:
corr_matrix = X_train_full.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.85)]

X_train_full = X_train_full.drop(columns=to_drop)
X_test = X_test.drop(columns=to_drop)

print(f"Dropped columns: {to_drop}")
print(f"Train shape after dropping: {X_train_full.shape}, Test shape: {X_test.shape}")

The cleaning process identified Income outliers which I now want to cap (to reduce skew without removing data points) if Income is in the selected features.

In [0]:
if 'Income' in X_train_full.columns:
       print(f"Income range before capping: [{X_train_full['Income'].min():.0f}, {X_train_full['Income'].max():.0f}]")
       
       # Calculate IQR 
       Q1 = X_train_full['Income'].quantile(0.25)
       Q3 = X_train_full['Income'].quantile(0.75)
       IQR = Q3 - Q1

       # Define outlier bounds
       train_lower_bound = Q1 - 1.5 * IQR
       train_upper_bound = Q3 + 1.5 * IQR

       X_train_full['Income'] = X_train_full['Income'].clip(upper=train_upper_bound, lower=train_lower_bound)
       X_test['Income'] = X_test['Income'].clip(upper=train_upper_bound, lower=train_lower_bound) # Apply same cap

       print(f"Income capped at bounds: [{train_lower_bound:.0f}, {train_upper_bound:.0f}]")
       print(f"Income range after capping: [{X_train_full['Income'].min():.0f}, {X_train_full['Income'].max():.0f}]")
else:
       print('Income not included in features')             

Evaluating two models, Random Forest and XGBoost. Stratified 5-fold cross-validation to preserve class distribution; handling `Response` imbalance with `class_weight='balanced'` in Random Forest and `scale_pos_weight` in XGBoost. Model evaluation using custom net profit scoring.

In [0]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}

models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=20,
        min_samples_leaf=10,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y_train_full == 0).sum() / (y_train_full == 1).sum(),
        random_state=42,
        verbosity=0
    )
}

for model_name, model in models.items():
    print(f"\n{'='*70}")
    print(f"Training {model_name} with cross-validation (profit optimised)")
    print(f"{'='*70}")
    
    scores = cross_validate(
        model, 
        X_train_full, 
        y_train_full, 
        cv=cv, 
        scoring=custom_profit_scorer,
        return_train_score=False
    )
    
    fold_profits = scores['test_score']
    
    print(f"Fold profits: {[f'{p:.2f}' for p in fold_profits]}")
    print(f"Mean max profit: {fold_profits.mean():.2f} (± {fold_profits.std():.2f})")
    
    cv_results[model_name] = {
        'mean_profit': fold_profits.mean(),
        'std_profit': fold_profits.std(),
        'fold_profits': fold_profits
    }

best_model_name = max(cv_results, key=lambda k: cv_results[k]['mean_profit'])
best_model_score = cv_results[best_model_name]['mean_profit']

print(f"\n{'='*70}")
print(f"Best model (CV): {best_model_name}")
print(f"Expected net profit per fold: {best_model_score:.2f}")
print(f"{'='*70}")


Train both models on full dataset and evaluate on the test set

In [0]:

trained_models = {}

for model_name, model_template in models.items():
    model = clone(model_template)
    model.fit(X_train_full, y_train_full)
    trained_models[model_name] = model

print(f"\n{'='*70}")
print(f"Model comparison on unseen test set")
print(f"{'='*70}")

test_results = {}
thresholds = np.arange(0.05, 0.95, 0.01)

for model_name, model in trained_models.items():
    print(f"\n{model_name}:")
    print("-" * 70)
    
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Find optimal threshold on test set
    test_profits = [calculate_net_profit(y_test, y_pred_proba, threshold=t) for t in thresholds]
    optimal_threshold = thresholds[np.argmax(test_profits)]
    max_profit = np.max(test_profits)
    
    # Apply optimal threshold
    y_pred = (y_pred_proba >= optimal_threshold).astype(int)
    
    # Calculate metrics
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc_roc = roc_auc_score(y_test, y_pred_proba)
    auc_pr = average_precision_score(y_test, y_pred_proba)
    
    # Store results and predictions for later use
    test_results[model_name] = {
        'model': model,  
        'optimal_threshold': optimal_threshold,
        'net_profit': max_profit,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc_roc': auc_roc,
        'auc_pr': auc_pr,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'cm': confusion_matrix(y_test, y_pred)
    }
    
    print(f"Optimal threshold: {optimal_threshold:.2f}")
    print(f"Net profit: {max_profit:.2f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall: {recall:.3f}")
    print(f"F1 score: {f1:.3f}")
    print(f"AUC-ROC: {auc_roc:.3f}")
    print(f"AUC-PR: {auc_pr:.3f}")
    print(f"\nConfusion matrix:")
    print(test_results[model_name]['cm'])

# Summary comparison
print(f"\n{'='*70}")
print(f"Model comparison summary")
print(f"{'='*70}\n")

summary_df = pd.DataFrame({
    'Model': list(test_results.keys()),
    'Optimal threshold': [test_results[m]['optimal_threshold'] for m in test_results],
    'Net profit': [test_results[m]['net_profit'] for m in test_results],
    'Precision': [test_results[m]['precision'] for m in test_results],
    'Recall': [test_results[m]['recall'] for m in test_results],
    'F1 score': [test_results[m]['f1'] for m in test_results],
    'AUC-ROC': [test_results[m]['auc_roc'] for m in test_results],
    'AUC-PR': [test_results[m]['auc_pr'] for m in test_results]
})

print(summary_df.to_string(index=False))

# Select best model
best_model_name = max(test_results, key=lambda k: test_results[k]['net_profit'])
best_profit = test_results[best_model_name]['net_profit']

print(f"\nBest model: {best_model_name}")
print(f"Net profit on test set: {best_profit:.2f}\n")

for model_name in test_results:
    profit = test_results[model_name]['net_profit']
    diff = profit - best_profit
    print(f"{model_name}:")
    print(f"   Profit: {profit:.2f} ({diff:+.2f} vs best)")

Visualisation of profitability against decision thresholds.

In [0]:
best_result = test_results[best_model_name]
y_pred_proba_best = best_result['y_pred_proba']
optimal_threshold_best = best_result['optimal_threshold']

profit_curve_values = [calculate_net_profit(y_test, y_pred_proba_best, threshold=t) for t in thresholds]

plt.figure(figsize=(12, 6))

plt.plot(thresholds, profit_curve_values, linewidth=2.5, label='Net profit potential', zorder=3)

optimal_idx = np.argmax(profit_curve_values)
optimal_thresh = thresholds[optimal_idx]
optimal_profit = profit_curve_values[optimal_idx]

plt.scatter([optimal_thresh], [optimal_profit],  s=150, zorder=5, 
            edgecolors='black', color='#2A9D8F', linewidths=1.5, label='Max profit')

plt.title('Profit vs decision threshold', fontsize=14, fontweight='bold')
plt.xlabel('Probability threshold (lower = more offers)', fontsize=12)
plt.ylabel('Net profit', fontsize=12)
plt.legend(loc='upper right', fontsize=10)
plt.xlim(0.05, 0.95)

# Add a text box with key insights for the business
textstr = f'Optimal threshold: {optimal_thresh:.2f}\nMax profit: {optimal_profit:,.0f}\nCost per offer: {COST_PER_OFFER:.2f}\nRevenue per acceptance: {REVENUE_PER_CONV:.2f}'
props = dict(boxstyle='round', facecolor='#D2B48C', alpha=0.7)
plt.text(0.20, 0.80, textstr, transform=plt.gca().transAxes, fontsize=10,
         verticalalignment='top',  bbox=props)

plt.tight_layout()
plt.savefig(output_dir / "Profit_vs_decision_threshold.png", dpi=150, bbox_inches='tight')
plt.show()

The above finds the decision threshold for maximum expected net profit, answering the exact business request. 

However, we can see from the above chart that there are **_nearly-as-profitable_ decision thresholds which offer to fewer customers** and therefore may have a **higher ROI**, and may be more desirable if, for instance, appetite for risk is lower and/or the budget for a campaign is limited.  

So, to answer the question of ROI and give the business the tools that it needs to answer the question **based on risk and budget considerations**, and also to provide a chart and table better-suited for all users, here is a version of the threshold/profitability that plots different decision thresholds (at 5% intervals for readability) with corresponding target list size (offers to make), conversion rate (percentage of offers accepted), resulting expected profit _and_ expected ROI, with accompanying table.

We can see that (at this 5% decision threshold granularity), the most profitable threshold is 0.35 with a profit of 273, but this has a cost of 225 and a ROI of 107%.  But at threshold 0.55, the predicted profit is 267 (only 6 less) with a ROI of 165%. 

In [0]:
# New results table including ROI
best_result = test_results[best_model_name]
y_pred_proba_best = best_result['y_pred_proba']
y_true = y_test.values
total_test_size = len(y_true)
actual_positives = int(y_true.sum())

thresholds = np.arange(0.30, 0.40, 0.01)
results = []

for thresh in thresholds:
    y_pred = (y_pred_proba_best >= thresh).astype(int)
    targeted_count = int(y_pred.sum())
    actual_conversions = int((y_pred & y_true).sum())
    precision = actual_conversions / targeted_count if targeted_count > 0 else 0
    total_cost = targeted_count * COST_PER_OFFER
    total_revenue = actual_conversions * REVENUE_PER_CONV
    net_profit = total_revenue - total_cost
    
    results.append({
        'Threshold': f"{thresh:.2f}",
        'Target list size': targeted_count,
        'Expected conversions': actual_conversions,
        'Conversion rate': round(precision * 100, 1),
        'Total cost': float(total_cost),
        'Total revenue': float(total_revenue),
        'Net profit': float(net_profit),
        'ROI': round(float(net_profit/total_cost)*100, 1)
    })

df_results = pd.DataFrame(results)

# Plotting
fig, ax = plt.subplots(figsize=(14, 8))

pale_sand = '#E8D8C0'
pomegranate_red = '#C41E3A'
profit_teal = '#2A9D8F'
slate_grey = '#596275'
goldenseed = '#DAA520'

x_vals = df_results['Threshold']

# Bar: Target list size
ax.yaxis.tick_left()
ax.tick_params(axis='y', left=True, right=False, labelleft=True, labelright=False)

ax.bar(x_vals, df_results['Target list size'], 
       color=pale_sand, edgecolor='#D2B48C', width=0.6, label='Target List Size (Offers)', alpha=0.9)

# Overlaid bar: expected conversions
ax.bar(x_vals, df_results['Expected conversions'], 
        color=pomegranate_red, edgecolor='#D2B48C', width=0.6,
        label='Expected conversions (acceptances)')

# Line: Net profit (right axis 1)
ax2 = ax.twinx()
ax2.yaxis.tick_right()
ax2.yaxis.set_label_position('right')
ax2.tick_params(axis='y', which='both', right=True, labelright=True,
                left=False, labelleft=False)
ax2.spines['right'].set_position(('outward', 0))
ax2.plot(x_vals, df_results['Net profit'], 
         color=profit_teal, marker='o', linewidth=3, markersize=8, 
         label='Net profit', zorder=15)

# Line: ROI (right axis 2, axis not visible)
ax3 = ax2.twinx()
ax3.spines['right'].set_position(('outward', 60))
ax3.yaxis.set_major_locator(plt.NullLocator())
ax3.yaxis.set_minor_locator(plt.NullLocator())
ax3.tick_params(axis='y', which='both', left=False, right=False,
                 labelleft=False, labelright=False)
ax3.spines['right'].set_visible(False)
ax3.set_ylabel('')
ax3.plot(x_vals, df_results['ROI'], label = 'ROI (%)',
         color=goldenseed, marker='o', linewidth=2.5, markersize=8, zorder=15)

# Labels
for i, row in df_results.iterrows():
    ax.text(row['Threshold'], row['Target list size'] + 3, 
            f"{row['Target list size']}", 
            ha='center', va='bottom', fontsize=9, fontweight='bold', color=slate_grey)

for i, row in df_results.iterrows():
    ax.text(row['Threshold'], row['Expected conversions'] + 3, 
            f"{row['Expected conversions']}\n({row['Conversion rate']:.0f}%)", 
            ha='center', va='bottom', fontsize=9, fontweight='bold', color=pomegranate_red)
    
for i, row in df_results.iterrows():
    ax2.text(row['Threshold'], row['Net profit'] + 3,
             f"{row['Net profit']:.0f}", 
             ha='center', va='bottom', fontsize=9, fontweight='bold', color=profit_teal)

for i, row in df_results.iterrows():
    ax3.text(row['Threshold'], row['ROI'] + 2, 
             f"{row['ROI']:.0f}%", 
             ha='center', va='bottom', fontsize=9, fontweight='bold', color='#FF8C00')

# Styling
ax.set_xlabel('Decision threshold', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of offers / offers accepted', fontsize=13, fontweight='bold')
ax.set_title('Marketing campaign impact for varying decision thresholds: offers, conversions, profit and ROI', fontsize=14, fontweight='bold')

ax2.set_ylabel('Net profit', fontsize=13, fontweight='bold', color=profit_teal)
ax2.tick_params(axis='y', labelcolor=profit_teal)

ax.grid(False)
ax2.grid(False)
ax3.grid(False)

# Legend
handles = [
    ax.containers[0],  # Target list size (bar container)
    ax.containers[1],  # Expected conversions (bar container)
    ax2.lines[0],      # Net profit line
    ax3.lines[0],      # ROI line
]
labels = [
    'Target list size (offers)',
    'Expected conversions (acceptances)',
    'Net profit',
    'ROI (%)'
]

ax.legend(handles, labels, loc='upper right', bbox_to_anchor=(0.85, 1), fontsize=10, frameon=True, framealpha=0.9)

# Subtitle
fig.suptitle(f'Based on test set: {total_test_size} Customers (actual positive responses: {actual_positives})', 
             fontsize=11, style='italic', y=1.02)

ax2.yaxis.tick_right()
ax2.tick_params(axis='y', which='both', right=True, labelright=True,
                left=False, labelleft=False)

plt.tight_layout()
# plt.savefig(output_dir / "Marketing_campaign_impact.png", dpi=150, bbox_inches='tight')
plt.show()

# Results table
print(df_results.to_string(index=False))

SHAP - feature importance

In [0]:
# SHAP with custom colours
xgb_model = test_results['XGBoost']['model']
explainer = shap.Explainer(xgb_model)
shap_values = explainer(X_test)

deep_ocean = '#004E7C'
warm_sand = '#D2B48C'
pomegranate_red = '#C41E3A'

# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)

# Recolour the bars
ax = plt.gca()
for patch in ax.patches:
    patch.set_facecolor(pomegranate_red)
    patch.set_edgecolor('#1a2d3a')
    patch.set_linewidth(0.5)

ax.set_title('SHAP feature importance (XGBoost)', fontsize=12, fontweight='bold', pad=20)
ax.set_xlabel('Mean |SHAP value|', fontsize=11)
plt.tight_layout()
plt.savefig(output_dir / "SHAP_feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()


**SHAP beeswarm chart**
This chart shows how each feature influences the model's prediction.
The vertical position shows the feature. The horizontal axis represents the impact on the model's predicted probability of offer acceptance (SHAP value).
- Right side (positive SHAP): Pushes the prediction up (more likely to accept)
- Left side (Negative SHAP): Pushes the prediction down (less likely to accept)
The colour: red = high feature value, blue = low feature value.
So, we cam see that for the most important feature, `Recency` (number of days since last purchase), most of the points to the right of the 0 axis are blue, showing that low recency (eg 2 days ago) is better than high recency (eg 365 days ago).

**Key insight: model limitations for new customers**. The SHAP analysis shows that the four most important features are `Recency`, `NumAccepted` (previous offers accepted), `DaysSinceJoin` (customer tenure) and `Total Spend` (spend over the past two years). However, these features are unavailable for new customers who have no history.

**Business implication:** This model is optimised for _existing_ customers with purchase history.
For offers to new customers, a separate model (or a modified version that excludes these features) would be required.

**Recommendation:** Develop a "new customer" model using only demographic and onboarding data at sign-up, eg age, income, education, marital status, household size information.

In [0]:
# SHAP beeswarm with custom colours
# fig, ax = plt.subplots(figsize=(10, 7))
fig = plt.figure(figsize=(10, 8))
# custom colormap: electric_blue (low) to pomegranate_red (high)
cmap = plt.cm.colors.LinearSegmentedColormap.from_list(
    'custom_shap', [deep_ocean, warm_sand, pomegranate_red]
)

shap.plots.beeswarm(shap_values, max_display=20, color=cmap)

# fig = plt.gcf()
fig.savefig(output_dir / "SHAP_beeswarm.png", dpi=150, bbox_inches='tight')
